In [0]:
# ==========================================================
# NOTEBOOK : 03_Silver
# PURPOSE  : Clean Bronze Data and Write Silver Delta
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *

print("Silver Notebook Started")

# ==========================================================
# STORAGE CONFIGURATION
# ==========================================================

storage_account = "retailadls67"
access_key = "bhy6cfJKxIz/zKE9cm/ofV8FACSlK06rqmC28MJPQSuWbFvYRiBKNs3j4s0pne0qo/erb+PxN+zx+AStajlJDg=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    access_key
)

spark.conf.set(
    "spark.databricks.delta.schema.autoMerge.enabled",
    "true"
)

# ==========================================================
# BRONZE PATHS
# ==========================================================

BRONZE_PRODUCTS = f"abfss://bronze@{storage_account}.dfs.core.windows.net/products"
BRONZE_ORDERS = f"abfss://bronze@{storage_account}.dfs.core.windows.net/orders"
BRONZE_CUSTOMERS = f"abfss://bronze@{storage_account}.dfs.core.windows.net/customers"
BRONZE_EXCHANGE = f"abfss://bronze@{storage_account}.dfs.core.windows.net/exchange_rates"

# ==========================================================
# SILVER PATHS
# ==========================================================

SILVER_PRODUCTS = f"abfss://silver@{storage_account}.dfs.core.windows.net/products"
SILVER_ORDERS = f"abfss://silver@{storage_account}.dfs.core.windows.net/orders"
SILVER_CUSTOMERS = f"abfss://silver@{storage_account}.dfs.core.windows.net/customers"
SILVER_EXCHANGE = f"abfss://silver@{storage_account}.dfs.core.windows.net/exchange_rates"

SILVER_REJECT_PRODUCTS = f"abfss://silver@{storage_account}.dfs.core.windows.net/reject_products"
SILVER_REJECT_ORDERS = f"abfss://silver@{storage_account}.dfs.core.windows.net/reject_orders"
SILVER_REJECT_CUSTOMERS = f"abfss://silver@{storage_account}.dfs.core.windows.net/reject_customers"

# ==========================================================
# READ BRONZE DELTA
# ==========================================================

products_df = spark.read.format("delta").load(BRONZE_PRODUCTS)
orders_df = spark.read.format("delta").load(BRONZE_ORDERS)
customers_df = spark.read.format("delta").load(BRONZE_CUSTOMERS)
exchange_df = spark.read.format("delta").load(BRONZE_EXCHANGE)

# ==========================================================
# DATA TYPE CONVERSIONS
# ==========================================================

products_df = products_df \
.withColumn("ProductID",col("ProductID").cast("int")) \
.withColumn("CostPrice",col("CostPrice").cast("double")) \
.withColumn("SellingPrice",col("SellingPrice").cast("double")) \
.withColumn("LaunchDate",to_date("LaunchDate"))

orders_df = orders_df \
.withColumn("OrderID",col("OrderID").cast("int")) \
.withColumn("CustomerID",col("CustomerID").cast("int")) \
.withColumn("ProductID",col("ProductID").cast("int")) \
.withColumn("Quantity",col("Quantity").cast("int")) \
.withColumn("UnitPrice",col("UnitPrice").cast("double")) \
.withColumn("Discount",col("Discount").cast("double")) \
.withColumn("TotalAmount",col("TotalAmount").cast("double")) \
.withColumn("OrderDate",to_date("OrderDate"))

customers_df = customers_df \
.withColumn("CustomerID",col("CustomerID").cast("int")) \
.withColumn("DOB",to_date("DOB")) \
.withColumn("JoinDate",to_date("JoinDate"))

# ==========================================================
# REMOVE DUPLICATES
# ==========================================================

products_df = products_df.dropDuplicates()
orders_df = orders_df.dropDuplicates()
customers_df = customers_df.dropDuplicates()

# ==========================================================
# HANDLE NULLS
# ==========================================================

products_df = products_df.fillna({"Status":"Unknown"})
customers_df = customers_df.fillna({"CustomerSegment":"Regular"})
orders_df = orders_df.fillna({"Discount":0})

# ==========================================================
# DATA VALIDATION
# ==========================================================

valid_products = products_df.filter(
    (col("SellingPrice") > col("CostPrice")) &
    (col("CostPrice") > 0)
)

reject_products = products_df.subtract(valid_products)

valid_orders = orders_df.filter(
    (col("Quantity") > 0) &
    (col("TotalAmount") > 0)
)

reject_orders = orders_df.subtract(valid_orders)

valid_customers = customers_df.filter(
    col("Email").isNotNull()
)

reject_customers = customers_df.subtract(valid_customers)

# ==========================================================
# WRITE REJECTS
# ==========================================================

reject_products.write.format("delta").mode("overwrite").save(SILVER_REJECT_PRODUCTS)
reject_orders.write.format("delta").mode("overwrite").save(SILVER_REJECT_ORDERS)
reject_customers.write.format("delta").mode("overwrite").save(SILVER_REJECT_CUSTOMERS)

# ==========================================================
# WRITE SILVER
# ==========================================================

valid_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(SILVER_PRODUCTS)

valid_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(SILVER_ORDERS)

valid_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(SILVER_CUSTOMERS)

# Keep exchange data in Silver as-is
exchange_df.write.format("delta").mode("overwrite").save(SILVER_EXCHANGE)

# ==========================================================
# VALIDATION
# ==========================================================

print("="*50)
print("Silver Layer Completed Successfully")
print("="*50)

print("Valid Products   :", valid_products.count())
print("Reject Products  :", reject_products.count())

print("Valid Orders     :", valid_orders.count())
print("Reject Orders    :", reject_orders.count())

print("Valid Customers  :", valid_customers.count())
print("Reject Customers :", reject_customers.count())

print("Exchange Records :", exchange_df.count())

valid_customers.display()

In [0]:
reject_products.display()